## Study 1: Gaussian noise


# Brain tumor segmentation robustness under synthetic MRI corruptions

**Study 1: additive Gaussian FLAIR noise.** Study 2 uses a positive multiplicative bias field.

This notebook compares unmitigated inputs, a fixed filter bank, and separately fine-tuned
models using a shared patient split. Severity and filters are selected on a 12-patient
validation subset; checkpoint selection uses the full validation set. Test predictions
are generated only after these decisions.

**Corrected pipeline: rerun required.** Original course documents are preserved under
`docs/historical/`; they do not validate this revised code.
This version changes masks, bias simulation, loss, augmentation seeds, and validation.
See the README for the experimental approach and project status.

Default data has 368 cases and gives 258/55/55 train/validation/test cases. Counts change
if verified pretraining IDs are excluded or a different dataset export is used. The test
split is held out from local fine-tuning; independence from bundle pretraining is unverified.

Use a CUDA GPU with substantial memory. Runtime depends on data, hardware and N4 correction;
the historical runtime estimate is not a benchmark for this corrected pipeline.


## 1. Setup

### 1.1 Environment install

See [README](../README.md) for the full setup. Run these commands from the repository root:

```bash
pip install "torch==2.3.1" --index-url https://download.pytorch.org/whl/cu121
pip install -r requirements.txt
```

Then set `BRATS_DATA_ROOT` to the directory directly containing the patient folders (see README -> Data).

### Imports

MONAI supplies bundle loading, sliding-window inference, Dice loss and HD95. Shared Python modules implement preprocessing, region conversion, input validation and paired statistics.


In [ ]:
from __future__ import annotations

import json, logging, random, time, zlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import nibabel as nib
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.stats import wilcoxon
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import monai
from monai.bundle import ConfigParser
from monai.inferers import sliding_window_inference
from monai.metrics import compute_hausdorff_distance


print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("monai", monai.__version__)


In [ ]:
import os
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path.cwd()


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print("Repository root:", REPO_ROOT)
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from brats_robustness.data import (
    build_manifest, make_split, load_or_create_split,
    load_4mod_volume as _load_volume, znorm_nonzero,
)
from brats_robustness.corruptions import (
    corrupt_gaussian_noise, gen_bias_field, corrupt_bias_field, make_filter_bank,
)
from brats_robustness.regions import probs_to_label_map, gt_to_regions, pred_to_regions
from brats_robustness.metrics import case_metrics, evaluate_condition, aggregate
from brats_robustness.inference import infer_volume as _infer_volume
from brats_robustness.training import make_region_loss
from brats_robustness.statistics import paired_comparisons, paired_vectors
from brats_robustness.reproducibility import experiment_identity, guard_run, guard_bundle, guard_prediction_dir

from brats_robustness.selection import select_severity

from brats_robustness.models import ensure_bundle


### Configuration

Set `BRATS_DATA_ROOT` directly to the directory containing patient folders. Run metadata
records configuration, dataset byte hashes, code hashes and installed package versions.
Incompatible caches raise an error; choose a fresh output root for changed experiments.
The bundle release is pinned to `0.5.3`; weights and inference configuration are hashed.
Data hashing requires a full dataset read per study.


In [ ]:
import os

@dataclass
class FineTuneConfig:
    n_steps: int = 3000
    log_every: int = 50
    val_every: int = 250            # val-DSC checkpoint check cadence
    batch_size: int = 2
    patch_size: Tuple[int, int, int] = (192, 192, 128)
    lr: float = 1e-5
    weight_decay: float = 1e-5
    noise_prob: float = 0.7
    noise_min: float = 0.2
    num_workers: int = 0


@dataclass
class Config:
    # ── Data paths ────────────────────────────────────────────────────────
    # Set the BRATS_DATA_ROOT environment variable to the folder that contains
    # MICCAI_BraTS2020_TrainingData (see README → Data). Defaults assume a
    # conventional local layout.
    train_root: Path = Path(
        os.environ.get(
            "BRATS_DATA_ROOT",
            "./BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData",
        )
    )
    out_root: Path = Path(os.environ.get("BRATS_OUT_ROOT", "./outputs/brats_robustness_v3"))

    # ── MONAI bundle ──────────────────────────────────────────────────────
    bundle_name: str = "brats_mri_segmentation"
    bundle_version: str = "0.5.3"
    bundle_revision: str = "6ea5b85f04151c71e5599a6f31000e5b042e8d80"
    bundle_dir: Path = Path(os.environ.get("BRATS_BUNDLE_DIR", "./monai_bundles/0.5.3"))

    # ── Reproducibility ───────────────────────────────────────────────────
    seed: int = 1337
    device: str = "cuda:0"

    # ── Split fractions ───────────────────────────────────────────────────
    train_frac: float = 0.70
    val_frac:   float = 0.15
    # test_frac is the remainder (~0.15)

    # ── Sliding-window inference ──────────────────────────────────────────
    roi_size: Tuple[int, int, int] = (240, 240, 160)
    sw_batch: int = 1
    sw_overlap: float = 0.5

    # ── Channel order (empirically verified against the bundle metadata) ──
    suffix_to_channel: Dict[str, int] = field(default_factory=lambda: {
        "t1ce": 0, "t1": 1, "t2": 2, "flair": 3,
    })

    # ── σ sweep (run on val subset, picks sigma_chosen) ───────────────────
    sweep_sigmas: Tuple[float, ...] = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0)
    sweep_n_val: int = 12   # number of val patients used in the sweep (cheap)

    # ── Filter bank for denoising mitigation ──────────────────────────────
    filter_names: Tuple[str, ...] = ("none", "mean_3", "gaussian_0p5", "gaussian_1p0", "gaussian_1p5")

    # ── Fine-tuning ───────────────────────────────────────────────────────
    fine_tune: FineTuneConfig = field(default_factory=FineTuneConfig)


CFG = Config()
if CFG.fine_tune.num_workers != 0:
    raise ValueError("Use num_workers=0 for deterministic dataset RNG in this notebook")

if not CFG.train_root.exists():
    raise FileNotFoundError(
        f"BraTS training data not found at {CFG.train_root}.\n"
        "Set the BRATS_DATA_ROOT environment variable to the folder containing "
        "MICCAI_BraTS2020_TrainingData (see README → Data)."
    )

# Create all output subdirectories up-front so cells that write are idempotent.
guard_run(CFG.out_root, experiment_identity(CFG, REPO_ROOT, "gaussian"))

for sub in [
    "manifests", "predictions", "metrics", "figures", "checkpoints",
    "predictions/clean_pretrained", "predictions/clean_finetuned",
    "predictions/sweep", "predictions/test_final",
]:
    (CFG.out_root / sub).mkdir(parents=True, exist_ok=True)
CFG.bundle_dir.mkdir(parents=True, exist_ok=True)

print("Output root:", CFG.out_root)
print("Bundle dir :", CFG.bundle_dir)


### Seeding

Python, NumPy and PyTorch are seeded. The dataset uses a private seeded RNG with a single-process DataLoader and a seeded shuffle generator. CUDA kernels may remain nondeterministic; exact numerical reproduction across devices is not promised.


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("brats_robust")

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)
device = torch.device(CFG.device if torch.cuda.is_available() else "cpu")
log.info("seed=%d device=%s", CFG.seed, device)


## 2. Manifest and seeded train / val / test split

### 2.1 Build manifest

A patient is included only if all five required modality files exist. Both `.nii` and `.nii.gz` files are supported.

In [ ]:
manifest = build_manifest(CFG.train_root)
log.info("Manifest: %d patients", len(manifest))
manifest.head()


### Patient-level split

Patient IDs are sorted before a seeded random split. ID-only assignments are persisted, checked against the current dataset and merged with current paths. This split is not stratified; tumor grade, size and acquisition-center balance are not guaranteed. It is held out from local fine-tuning, with pretraining overlap unverified unless an independent identity audit is supplied.


In [ ]:
SPLIT_CSV = CFG.out_root / "manifests" / f"split_seed{CFG.seed}.csv"

split_df = load_or_create_split(manifest, SPLIT_CSV, CFG.train_frac, CFG.val_frac, CFG.seed)
print(split_df["split"].value_counts())
train_pids = split_df[split_df["split"] == "train"]["pid"].tolist()
val_pids   = split_df[split_df["split"] == "val"]["pid"].tolist()
test_pids  = split_df[split_df["split"] == "test"]["pid"].tolist()
train_df = split_df[split_df["split"] == "train"].reset_index(drop=True)
val_df   = split_df[split_df["split"] == "val"].reset_index(drop=True)
test_df  = split_df[split_df["split"] == "test"].reset_index(drop=True)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")


## Pretrained model

The MONAI `brats_mri_segmentation` release is pinned to `0.5.3`. Its model card describes
BraTS 2018 training and a SegResNet approach related to the BraTS2018 winning method.
This is not a claim to have trained the winning architecture from scratch.

We qualify `SegResNet` in the inference config if required, verify channel order and metadata
version, and load weights strictly. Weight/configuration hashes are recorded with every run.
The test split is independent of local fine-tuning; independence from pretraining is unverified.


In [ ]:
bundle_path = CFG.bundle_dir / CFG.bundle_name
ensure_bundle(bundle_path, CFG.bundle_name, CFG.bundle_revision)

# Patch inference.json: fully qualify SegResNet target so ConfigParser can resolve it
inference_cfg_path = bundle_path / "configs" / "inference.json"
cfg_json = json.loads(inference_cfg_path.read_text())
if cfg_json.get("network_def", {}).get("_target_") == "SegResNet":
    cfg_json["network_def"]["_target_"] = "monai.networks.nets.SegResNet"
    inference_cfg_path.write_text(json.dumps(cfg_json, indent=4))
    log.info("Patched network_def._target_ to monai.networks.nets.SegResNet")
else:
    log.info("inference.json already patched")
log.info("Bundle ready at %s", bundle_path)

guard_bundle(CFG.out_root, bundle_path, CFG.bundle_version)


### 3.1 **Channel-order assertion** (do not skip)

This cell reads the bundle's `metadata.json`, extracts the declared input channel definitions, and asserts they match `CFG.suffix_to_channel`. The bundle declares `(T1c, T1, T2, FLAIR)` at indices `(0, 1, 2, 3)`. If a future bundle revision changes the order, this assertion fails loudly — update `CFG.suffix_to_channel` rather than bypassing the check.

In [ ]:
metadata = json.loads((bundle_path / "configs" / "metadata.json").read_text())
declared = metadata["network_data_format"]["inputs"]["image"]["channel_def"]
out_def  = metadata["network_data_format"]["outputs"]["pred"]["channel_def"]
log.info("Input  channel_def: %s", declared)
log.info("Output channel_def: %s", out_def)

def norm_modname(s: str) -> str:
    s = s.lower().replace("-", "").replace(" ", "")
    if "flair" in s: return "flair"
    if "t1c" in s or "t1gd" in s: return "t1ce"
    if s == "t1" or s.startswith("t1w"): return "t1"
    if "t2" in s: return "t2"
    return s

declared_norm = {int(k): norm_modname(v) for k, v in declared.items()}
expected_norm = {idx: suf for suf, idx in CFG.suffix_to_channel.items()}
assert declared_norm == expected_norm, (
    f"Bundle channel order {declared_norm} differs from CFG.suffix_to_channel {expected_norm}. "
    "Update CFG.suffix_to_channel rather than bypassing this assert."
)
log.info("✓ channel order verified — FLAIR is channel %d", CFG.suffix_to_channel["flair"])


### 3.2 Load network and weights

In [ ]:
parser = ConfigParser(); parser.read_config(str(inference_cfg_path))
network = parser.get_parsed_content("network_def").to(device)
state = torch.load(bundle_path / "models" / "model.pt", map_location=device, weights_only=False)
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]
missing, unexpected = network.load_state_dict(state, strict=True)
log.info("Weights loaded — missing=%d, unexpected=%d", len(missing), len(unexpected))
network.eval()


## Volume loading and inference

Shared helpers check 3D shape, finite intensities, co-registration, label geometry and 1mm isotropic spacing, then normalize nonzero voxels per channel. The input must be the standard preprocessed BraTS2020 export; arbitrary clinical scans need a separate preprocessing pipeline. Sliding-window inference returns independent sigmoid probabilities in TC/WT/ET order.


In [ ]:
def load_4mod_volume(row):
    return _load_volume(row, CFG.suffix_to_channel)


def infer_volume(net, vol):
    return _infer_volume(net, vol, CFG, device)


def save_nifti(arr, affine, out_path):
    # Write in the same directory and atomically publish a complete prediction.
    out_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = out_path.with_name(out_path.name.replace(".nii.gz", ".partial.nii.gz"))
    nib.save(nib.Nifti1Image(arr, affine), str(temporary))
    temporary.replace(out_path)


def patient_noise_seed(pid):
    # Full CRC32 avoids the original modulo-10000 seed collisions.
    return CFG.seed + zlib.crc32(pid.encode())


## Gaussian corruption

Add zero-mean Gaussian noise to the clean nonzero FLAIR support. Severity is relative to the clean support standard deviation. `CFG.seed + crc32(pid)` is a stable patient seed, shared across evaluation arms. Filters receive the clean support mask explicitly and retain negative noisy tissue values.


In [ ]:
# Gaussian corruption is implemented in brats_robustness/corruptions.py.


### Shared filter bank

SciPy filters use fixed kernels and preserve the clean input support mask.


In [ ]:
FILTER_BANK = make_filter_bank(include_n4=False)
log.info("Filter bank: %s", list(FILTER_BANK))


## 6. Metrics

Per-case per-region: **DSC, HD95 (mm), sensitivity (recall), FPvol (mL), FNvol (mL)** for **WT / TC / ET**.

- **HD95** is the 95th-percentile Hausdorff distance via `monai.metrics.compute_hausdorff_distance`. We pass real voxel spacing from the GT NIfTI header so units are mm.
- **Empty masks**: if both pred and GT are empty, DSC=1, HD95=0 (trivially perfect). If exactly one is empty, DSC=0, HD95 is set to the volume diagonal in mm (worst case) so the aggregate is not `nan`-poisoned.
- **FPvol / FNvol** are in mL: `voxels × |det(affine[:3,:3])| / 1000`.

We work off the predicted *label map* (not the raw probabilities) so the metric is reproducible from the cached NIfTI files alone.

In [ ]:
# Region conversion and metrics are shared between both studies.
# Empty-mask HD95 uses the volume diagonal, as described in the README.


## 7. Inference orchestrators

These two helpers do all the actual GPU work. Both write predictions as NIfTI files; both are **idempotent** — if the output already exists they skip the patient.

- `infer_subset_clean(net, df_subset, out_dir)` — run `net` on the four-modality clean volumes.
- `infer_subset_corrupted(net, df_subset, out_dir, sigma_rel, filter_fn)` — corrupt FLAIR with the per-patient seed, optionally apply a denoising `filter_fn` to the noisy FLAIR, then run inference.

In [ ]:
def infer_subset_clean(net: torch.nn.Module, df_subset: pd.DataFrame,
                       out_dir: Path, desc: str) -> None:
    """Run net on the clean 4-mod volume for each patient in df_subset. Idempotent."""
    guard_prediction_dir(out_dir, net, {"kind": "clean"})
    for _, row in tqdm(list(df_subset.iterrows()), desc=desc):
        pid = row["pid"]
        out_path = out_dir / f"{pid}_pred.nii.gz"
        if out_path.exists():
            continue
        vol, affine = load_4mod_volume(row)
        probs = infer_volume(net, vol)
        save_nifti(probs_to_label_map(probs), affine, out_path)


def infer_subset_corrupted(net: torch.nn.Module, df_subset: pd.DataFrame,
                           out_dir: Path, desc: str,
                           sigma_rel: float,
                           filter_fn: Callable[[np.ndarray, np.ndarray], np.ndarray]) -> None:
    """Per patient: corrupt FLAIR (per-patient seed), apply filter_fn to corrupted FLAIR, infer.
    Identity filter_fn means 'no preprocessing', i.e. the raw corrupted-FLAIR condition.
    """
    guard_prediction_dir(out_dir, net, {"kind": "gaussian", "severity": float(sigma_rel), "preprocessing": filter_fn.preprocessing_name})
    flair_idx = CFG.suffix_to_channel["flair"]
    for _, row in tqdm(list(df_subset.iterrows()), desc=desc):
        pid = row["pid"]
        out_path = out_dir / f"{pid}_pred.nii.gz"
        if out_path.exists():
            continue
        vol, affine = load_4mod_volume(row)
        noisy = corrupt_gaussian_noise(vol[flair_idx], sigma_rel, seed=patient_noise_seed(pid))
        brain_mask = vol[flair_idx] != 0
        filtered = filter_fn(noisy, brain_mask)
        vol[flair_idx] = filtered
        probs = infer_volume(net, vol)
        save_nifti(probs_to_label_map(probs), affine, out_path)


## Validation severity sweep

Sweep relative noise levels 0.5 through 3.0 on the first 12 validation patients (sorted by
patient ID). Select the smallest severity yielding an 0.08 absolute WT-DSC drop in the
**unmitigated** condition. If none reaches the target, use the maximum observed drop and
record the fallback. Stop if clean validation WT-DSC is below 0.85.

This rule is independent of mitigation scores. Separately choose the filter with the highest
validation mean WT-DSC at the selected severity, then freeze choices before test evaluation.


In [ ]:
SWEEP_VAL_DF = val_df.iloc[:CFG.sweep_n_val].reset_index(drop=True)
SWEEP_DIR = CFG.out_root / "predictions" / "sweep"
CLEAN_VAL_DIR = SWEEP_DIR / "clean"
log.info("Sweep on %d val patients, sigmas=%s", len(SWEEP_VAL_DF), CFG.sweep_sigmas)

# (a) Clean inference on the sweep val subset — reused across all sigmas.
infer_subset_clean(network, SWEEP_VAL_DF, CLEAN_VAL_DIR, "sweep clean")

# (b) For each sigma, for each filter (including "none"), infer + save.
for sigma in CFG.sweep_sigmas:
    sigma_tag = f"sigma_{sigma:.2f}".replace(".", "p")
    for fname in CFG.filter_names:
        out_dir = SWEEP_DIR / sigma_tag / fname
        infer_subset_corrupted(
            net=network, df_subset=SWEEP_VAL_DF,
            out_dir=out_dir, desc=f"sweep {sigma_tag} / {fname}",
            sigma_rel=sigma, filter_fn=FILTER_BANK[fname],
        )

log.info("Sweep inference done — predictions cached under %s", SWEEP_DIR)


### 8.1 Score the sweep — pick `sigma_chosen`

In [ ]:
# Score clean once, plus every (sigma, filter) pair.
sweep_rows = []
clean_eval = evaluate_condition(SWEEP_VAL_DF, CLEAN_VAL_DIR, "Clean")
clean_eval["sigma"] = np.nan
clean_eval["filter"] = "clean"
sweep_rows.append(clean_eval)

for sigma in CFG.sweep_sigmas:
    sigma_tag = f"sigma_{sigma:.2f}".replace(".", "p")
    for fname in CFG.filter_names:
        out_dir = SWEEP_DIR / sigma_tag / fname
        e = evaluate_condition(SWEEP_VAL_DF, out_dir, f"sigma{sigma}_filter_{fname}")
        e["sigma"] = sigma; e["filter"] = fname
        sweep_rows.append(e)

sweep_df = pd.concat(sweep_rows, ignore_index=True)
sweep_df.to_csv(CFG.out_root / "metrics" / "sweep_per_case.csv", index=False)

# Mean WT DSC per (sigma, filter) for the decision rule
wt = sweep_df[sweep_df["region"] == "WT"]
clean_wt_mean = wt[wt["filter"] == "clean"]["DSC"].mean()
log.info("Sweep clean WT DSC mean = %.4f", clean_wt_mean)

summary_rows = []
for sigma in CFG.sweep_sigmas:
    for fname in CFG.filter_names:
        sel = wt[(wt["filter"] == fname) & (wt["sigma"] == sigma)]
        if sel.empty:
            continue
        summary_rows.append({"sigma": sigma, "filter": fname,
                             "DSC_WT_mean": sel["DSC"].mean(),
                             "DSC_WT_std":  sel["DSC"].std()})
sweep_summary = pd.DataFrame(summary_rows)
sweep_summary.to_csv(CFG.out_root / "metrics" / "sweep_summary.csv", index=False)
print(sweep_summary.pivot(index="sigma", columns="filter", values="DSC_WT_mean").round(3))


In [ ]:
corrupted_validation_means = [
    float(wt[(wt["filter"] == "none") & (wt["sigma"] == s)].DSC.mean())
    for s in CFG.sweep_sigmas
]
clean_wt_mean = float(wt[wt["filter"] == "clean"].DSC.mean())
sigma_chosen, severity_reason = select_severity(
    CFG.sweep_sigmas, clean_wt_mean, corrupted_validation_means,
)
(CFG.out_root / "metrics" / "sigma_chosen.txt").write_text(str(sigma_chosen))
(CFG.out_root / "metrics" / "severity_selection.json").write_text(
    json.dumps({"chosen": sigma_chosen, "reason": severity_reason, "target_drop": 0.08}, indent=2)
)
log.info("Chosen sigma=%s: %s", sigma_chosen, severity_reason)


### 8.2 Plot DSC vs σ from the sweep

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(clean_wt_mean, color="gray", ls="--", label=f"clean (mean={clean_wt_mean:.3f})")
for fname in CFG.filter_names:
    sub = sweep_summary[sweep_summary["filter"] == fname].sort_values("sigma")
    ax.errorbar(sub["sigma"], sub["DSC_WT_mean"], yerr=sub["DSC_WT_std"],
                marker="o", label=fname, capsize=3)
ax.axvline(sigma_chosen, color="red", ls=":", alpha=0.6, label=f"chosen σ={sigma_chosen}")
ax.set_xlabel("FLAIR noise σ (× FLAIR std)")
ax.set_ylabel("mean WT DSC")
ax.set_ylim(0, 1)
ax.set_title(f"σ sweep on val (n={len(SWEEP_VAL_DF)}): each filter vs corruption severity")
ax.grid(alpha=0.3); ax.legend(loc="best", fontsize=9)
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "sweep_DSC_WT_vs_sigma.png", dpi=140)
plt.show()


## Gaussian-augmented fine-tuning

Fine-tune on training patients using AdamW (learning rate 1e-5, weight decay 1e-5), 3000
steps, batch size 2, patches 192x192x128. With probability 0.7 add Gaussian noise with
relative severity sampled uniformly from the configured minimum to the selected severity.
Keep the other samples clean. Use sigmoid Dice plus independent binary cross entropy for
all three overlapping tumor regions. AMP is enabled on CUDA only.

Validate every 250 steps on the **full validation set**, using consistent nested region
conversion. Save the checkpoint with highest mean corrupted WT-DSC. Clean test evaluation
measures changes in clean performance rather than presuming there is no forgetting.


In [ ]:
class BraTSPatchDataset(torch.utils.data.Dataset):
    """Random brain-centered patches with FLAIR noise augmentation.

    Each __getitem__ returns (vol[4, ph, pw, pd] float32 normalized,
                              gt_regions[3, ph, pw, pd] float32 in TC/WT/ET).
    With probability `noise_prob`, FLAIR is corrupted with sigma_rel ~ Uniform(0.2, sigma_max).
    The augmentation seed is drawn from the dataset's own RNG, so training noise varies
    across steps but the whole loop is reproducible from `random_state`.
    """

    def __init__(self, df: pd.DataFrame, patch_size: Tuple[int, int, int],
                 noise_prob: float, sigma_max: float, random_state: int):
        self.df = df.reset_index(drop=True)
        self.patch = patch_size
        self.noise_prob = noise_prob
        self.noise_lo = CFG.fine_tune.noise_min
        self.noise_hi = sigma_max
        self.rng = np.random.default_rng(random_state)

    def __len__(self) -> int:
        return len(self.df)

    @staticmethod
    def _seg_to_regions(seg: np.ndarray) -> np.ndarray:
        return np.stack([
            (seg == 1) | (seg == 4),
            seg > 0,
            seg == 4,
        ], axis=0).astype(np.float32)

    def _random_patch(self, vol: np.ndarray, gt_r: np.ndarray):
        H, W, D = vol.shape[1:]
        ph, pw, pd_ = self.patch
        brain = vol[CFG.suffix_to_channel["flair"]] != 0
        if brain.any():
            zs = np.argwhere(brain)
            c = zs[self.rng.integers(0, len(zs))]
            h0 = int(np.clip(c[0] - ph // 2, 0, H - ph))
            w0 = int(np.clip(c[1] - pw // 2, 0, W - pw))
            d0 = int(np.clip(c[2] - pd_ // 2, 0, D - pd_))
        else:
            h0 = int(self.rng.integers(0, max(1, H - ph + 1)))
            w0 = int(self.rng.integers(0, max(1, W - pw + 1)))
            d0 = int(self.rng.integers(0, max(1, D - pd_ + 1)))
        v = vol[:, h0:h0 + ph, w0:w0 + pw, d0:d0 + pd_]
        g = gt_r[:, h0:h0 + ph, w0:w0 + pw, d0:d0 + pd_]
        return v, g

    def __getitem__(self, i: int):
        row = self.df.iloc[i]
        vol, _ = load_4mod_volume(row)
        seg = nib.load(row["seg"]).get_fdata().astype(np.int16)
        if self.rng.random() < self.noise_prob:
            sigma_rel = float(self.rng.uniform(self.noise_lo, self.noise_hi))
            aug_seed = int(self.rng.integers(0, 2 ** 31 - 1))
            vol[CFG.suffix_to_channel["flair"]] = corrupt_gaussian_noise(
                vol[CFG.suffix_to_channel["flair"]], sigma_rel, seed=aug_seed
            )
        vol = znorm_nonzero(vol)
        gt_r = self._seg_to_regions(seg)
        v, g = self._random_patch(vol, gt_r)
        return torch.from_numpy(v).float(), torch.from_numpy(g).float()


### Checkpoint selection

At each validation interval, evaluate all validation patients under the selected corruption and save the checkpoint with highest mean WT-DSC. Validation and test use the same region-conversion logic.


In [ ]:
def _quick_val_dsc(net: torch.nn.Module, df_val_subset: pd.DataFrame, sigma_rel: float) -> float:
    """Return mean WT-DSC of `net` on `df_val_subset` under sigma_rel FLAIR corruption (no filter).

    No NIfTI written — used as a per-checkpoint validation signal during fine-tuning.
    """
    net.eval()
    flair_idx = CFG.suffix_to_channel["flair"]
    dscs = []
    with torch.no_grad():
        for _, row in df_val_subset.iterrows():
            vol, _ = load_4mod_volume(row)
            vol[flair_idx] = corrupt_gaussian_noise(vol[flair_idx], sigma_rel,
                                                   seed=patient_noise_seed(row["pid"]))
            probs = infer_volume(net, vol)
            seg = nib.load(row["seg"]).get_fdata().astype(np.int16)
            p_wt = probs_to_label_map(probs) > 0
            g_wt = seg > 0
            inter = np.logical_and(p_wt, g_wt).sum()
            denom = p_wt.sum() + g_wt.sum()
            dscs.append(2 * inter / denom if denom > 0 else 1.0)
    return float(np.mean(dscs))


# Rebuild fresh fine-tunable network from bundle weights so the cell is idempotent.
log.info("Building fine-tuneable network from bundle weights")
parser_ft = ConfigParser(); parser_ft.read_config(str(inference_cfg_path))
net_ft = parser_ft.get_parsed_content("network_def").to(device)
state_ft = torch.load(bundle_path / "models" / "model.pt", map_location=device, weights_only=False)
if isinstance(state_ft, dict) and "state_dict" in state_ft:
    state_ft = state_ft["state_dict"]
net_ft.load_state_dict(state_ft, strict=True)

CHECKPOINT_BEST = CFG.out_root / "checkpoints" / "best_finetuned.pt"
LOSS_HISTORY_NPZ = CFG.out_root / "checkpoints" / "loss_history.npz"

ft_cfg = CFG.fine_tune
train_ds = BraTSPatchDataset(
    train_df, patch_size=ft_cfg.patch_size,
    noise_prob=ft_cfg.noise_prob, sigma_max=float(sigma_chosen),
    random_state=CFG.seed,
)
loader = torch.utils.data.DataLoader(
    train_ds, batch_size=ft_cfg.batch_size, shuffle=True,
    num_workers=ft_cfg.num_workers, pin_memory=True, drop_last=True,
    generator=torch.Generator().manual_seed(CFG.seed),
)
loader_iter = iter(loader)

optimizer = torch.optim.AdamW(net_ft.parameters(), lr=ft_cfg.lr, weight_decay=ft_cfg.weight_decay)
loss_fn = make_region_loss()
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

val_subset_for_ft = val_df.reset_index(drop=True)

if CHECKPOINT_BEST.exists() and LOSS_HISTORY_NPZ.exists():
    log.info("Found cached fine-tune outputs — skipping training. Delete %s to retrain.", CHECKPOINT_BEST)
    history = np.load(LOSS_HISTORY_NPZ, allow_pickle=True)
    losses = history["losses"].tolist()
    val_dsc_history = history["val_dsc_history"].tolist()
else:
    net_ft.train()
    losses, val_dsc_history = [], []
    best_val = -1.0
    pbar = tqdm(range(1, ft_cfg.n_steps + 1), desc="fine-tune")
    for step in pbar:
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            logits = net_ft(x)
            loss = loss_fn(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        losses.append(float(loss.item()))
        if step % ft_cfg.log_every == 0:
            pbar.set_postfix(loss=f"{np.mean(losses[-ft_cfg.log_every:]):.4f}")
        if step % ft_cfg.val_every == 0 or step == ft_cfg.n_steps:
            v = _quick_val_dsc(net_ft, val_subset_for_ft, sigma_rel=float(sigma_chosen))
            val_dsc_history.append((step, v))
            log.info("step %d val WT-DSC (σ=%s) = %.4f  best=%.4f", step, sigma_chosen, v, best_val)
            if v > best_val:
                best_val = v
                torch.save(net_ft.state_dict(), CHECKPOINT_BEST)
                log.info("  → new best, saved")
            net_ft.train()
    np.savez(LOSS_HISTORY_NPZ, losses=np.array(losses), val_dsc_history=np.array(val_dsc_history, dtype=object))

# Load best weights into net_ft for downstream use
net_ft.load_state_dict(torch.load(CHECKPOINT_BEST, map_location=device, weights_only=False))
net_ft.eval()
log.info("Fine-tune done. Best checkpoint loaded.")


### 9.2 Loss + val-DSC curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve with running mean
losses_arr = np.array(losses)
ax1.plot(losses_arr, color="lightgray", alpha=0.5, label="step loss")
if len(losses_arr) >= 50:
    k = 50
    running = np.convolve(losses_arr, np.ones(k) / k, mode="valid")
    ax1.plot(np.arange(k - 1, len(losses_arr)), running, color="C0", label=f"running mean ({k})")
ax1.set_xlabel("step"); ax1.set_ylabel("DiceCE loss"); ax1.set_title("Fine-tune loss")
ax1.grid(alpha=0.3); ax1.legend()

# Val DSC curve
if val_dsc_history:
    steps_v = [int(s) for s, _ in val_dsc_history]
    vals_v  = [float(v) for _, v in val_dsc_history]
    ax2.plot(steps_v, vals_v, marker="o")
    ax2.set_xlabel("step"); ax2.set_ylabel(f"val WT-DSC @ σ={sigma_chosen}")
    ax2.set_title("Val WT-DSC (corrupted) during fine-tune")
    ax2.set_ylim(0, 1); ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "finetune_curves.png", dpi=140)
plt.show()


In [ ]:
# Select the filter from validation results before touching the held-out test set.
validation_filter_names = [name for name in CFG.filter_names if name != "none"]
validation_filters = sweep_summary[
    (sweep_summary["sigma"] == sigma_chosen)
    & (sweep_summary["filter"].isin(validation_filter_names))
]
assert not validation_filters.empty, "No validation filter scores found for sigma_chosen."
selected_filter_name = validation_filters.loc[
    validation_filters["DSC_WT_mean"].idxmax(), "filter"
]
best_filter_cond = f"Filter {selected_filter_name}"
(CFG.out_root / "metrics" / "selected_filter.txt").write_text(best_filter_cond)
log.info("Selected filter from validation: %s", best_filter_cond)


## 10. Final evaluation on the held-out test set

This is the only place the test split is used. We evaluate the following conditions on all `len(test_df)` ≈ 55 test patients:

1. **Clean (pretrained)** — model + clean inputs. Sanity baseline.
2. **Corrupted (pretrained)** — model + FLAIR noised at `sigma_chosen`. The "broken" condition.
3. **Filtered × Corrupted (pretrained)** — for each filter ∈ {`mean_3`, `gaussian_0p5`, `gaussian_1p0`, `gaussian_1p5`}: corrupt then filter then infer.
4. **Corrupted (fine-tuned)** — fine-tuned model + FLAIR noised at `sigma_chosen`. The primary mitigation.
5. **Clean (fine-tuned)** — fine-tuned model + clean inputs. *Forgetting check* — measure clean-domain change.

Total: 1 + 1 + 4 + 1 + 1 = **8 conditions × 55 patients = 440 inferences**. All cached to disk.

In [ ]:
TEST_DIR = CFG.out_root / "predictions" / "test_final"
sigma_chosen_float = float(sigma_chosen)

# 1. Clean (pretrained)
infer_subset_clean(network, test_df, TEST_DIR / "clean_pretrained", "test: clean pretrained")

# 2. Corrupted (pretrained), no filter
infer_subset_corrupted(
    network, test_df, TEST_DIR / "corrupted_pretrained",
    "test: corrupted pretrained", sigma_rel=sigma_chosen_float,
    filter_fn=FILTER_BANK["none"],
)

# 3. Each filter on corrupted (pretrained)
for fname in CFG.filter_names:
    if fname == "none":
        continue
    infer_subset_corrupted(
        network, test_df, TEST_DIR / f"filter_{fname}",
        f"test: filter {fname}", sigma_rel=sigma_chosen_float,
        filter_fn=FILTER_BANK[fname],
    )

# 4. Corrupted (fine-tuned)
infer_subset_corrupted(
    net_ft, test_df, TEST_DIR / "corrupted_finetuned",
    "test: corrupted finetuned", sigma_rel=sigma_chosen_float,
    filter_fn=FILTER_BANK["none"],
)

# 5. Clean (fine-tuned)
infer_subset_clean(net_ft, test_df, TEST_DIR / "clean_finetuned", "test: clean finetuned")
log.info("All test predictions saved under %s", TEST_DIR)


### 10.1 Per-case metrics and aggregate table

In [ ]:
COND_DIRS = {
    "Clean (pretrained)":         TEST_DIR / "clean_pretrained",
    "Corrupted (pretrained)":     TEST_DIR / "corrupted_pretrained",
    "Filter mean_3":              TEST_DIR / "filter_mean_3",
    "Filter gaussian_0p5":        TEST_DIR / "filter_gaussian_0p5",
    "Filter gaussian_1p0":        TEST_DIR / "filter_gaussian_1p0",
    "Filter gaussian_1p5":        TEST_DIR / "filter_gaussian_1p5",
    "Corrupted (fine-tuned)":     TEST_DIR / "corrupted_finetuned",
    "Clean (fine-tuned)":         TEST_DIR / "clean_finetuned",
}

per_case_rows = []
for cond, d in COND_DIRS.items():
    per_case_rows.append(evaluate_condition(test_df, d, cond))
test_per_case = pd.concat(per_case_rows, ignore_index=True)
test_per_case.to_csv(CFG.out_root / "metrics" / "test_per_case.csv", index=False)

test_agg = aggregate(test_per_case)
test_agg.to_csv(CFG.out_root / "metrics" / "test_aggregate.csv", index=False)

# Compact view: DSC mean ± std per (condition, region)
view = test_agg.pivot(index="condition", columns="region", values=["DSC_mean", "DSC_std"])
view = view.swaplevel(0, 1, axis=1).sort_index(axis=1)
print("\n=== Test set DSC summary ===")
print(view.round(3))
test_agg


### 10.2 Bar chart — mean DSC per condition per region

In [ ]:
regions = ["WT", "TC", "ET"]
conditions_order = list(COND_DIRS.keys())
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(conditions_order))
width = 0.27

for i, region in enumerate(regions):
    sub = test_agg[test_agg["region"] == region].set_index("condition")
    means = sub.loc[conditions_order, "DSC_mean"].values
    stds  = sub.loc[conditions_order, "DSC_std"].values
    ax.bar(x + (i - 1) * width, means, width, yerr=stds, label=region, capsize=2)

ax.set_xticks(x); ax.set_xticklabels(conditions_order, rotation=30, ha="right")
ax.set_ylabel("DSC (mean ± std)"); ax.set_ylim(0, 1)
ax.set_title(f"Test set DSC by condition and region (n={len(test_df)}, σ={sigma_chosen})")
ax.grid(alpha=0.3, axis="y"); ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "test_dsc_bar.png", dpi=140)
plt.show()

# Selection timing: The filter is selected from validation results before final test inference.


### Paired statistics

Join conditions by unique patient ID. Report two-sided Wilcoxon p-values, Holm-adjusted p-values across the planned comparisons within each study, paired mean/median differences, and a 5000-resample patient bootstrap CI for the mean difference. A nonsignificant result is not an equivalence finding.


In [ ]:
def wt_dsc_vec(df: pd.DataFrame, cond: str) -> np.ndarray:
    sub = df[(df["condition"] == cond) & (df["region"] == "WT")].sort_values("pid")
    return sub["DSC"].to_numpy()


# Required pairs
pairs = [
    ("Corrupted vs Clean (pretrained)",       "Corrupted (pretrained)", "Clean (pretrained)"),
    (f"Best filter ({best_filter_cond}) vs Corrupted (pretrained)", best_filter_cond, "Corrupted (pretrained)"),
    ("Fine-tune vs best filter",              "Corrupted (fine-tuned)", best_filter_cond),
    ("Forgetting: Clean (fine-tuned) vs Clean (pretrained)", "Clean (fine-tuned)", "Clean (pretrained)"),
]

wcx_rows = paired_comparisons(test_per_case, pairs, expected_n=len(test_df), seed=CFG.seed)
wcx_df = pd.DataFrame(wcx_rows)
wcx_df.to_csv(CFG.out_root / "metrics" / "test_wilcoxon_WT_DSC.csv", index=False)
wcx_df


### 10.4 Qualitative figures — two representative cases

We pick two test patients with **non-trivial tumor volume** (so the slice is informative) and render: clean / corrupted / best-filter / fine-tuned predictions overlaid on FLAIR at the slice with the largest GT WT area.

In [ ]:
def render_test_case(pid: str, out_png: Path) -> None:
    """Render the 4 conditions side-by-side at the max-tumor axial slice."""
    row = test_df[test_df["pid"] == pid].iloc[0]
    flair = nib.load(row["flair"]).get_fdata()
    gt = nib.load(row["seg"]).get_fdata().astype(np.int16)
    z = int(np.argmax((gt > 0).sum(axis=(0, 1))))

    panels = [
        ("Clean (pretrained)",    TEST_DIR / "clean_pretrained"   / f"{pid}_pred.nii.gz"),
        ("Corrupted (pretrained)",TEST_DIR / "corrupted_pretrained"/ f"{pid}_pred.nii.gz"),
        (best_filter_cond,        COND_DIRS[best_filter_cond]     / f"{pid}_pred.nii.gz"),
        ("Corrupted (fine-tuned)",TEST_DIR / "corrupted_finetuned"/ f"{pid}_pred.nii.gz"),
    ]
    fig, axes = plt.subplots(1, len(panels) + 1, figsize=(3.0 * (len(panels) + 1), 3.5))
    # GT panel
    axes[0].imshow(flair[:, :, z].T, cmap="gray", origin="lower")
    g_wt = (gt > 0)[:, :, z]; g_tc = ((gt == 1) | (gt == 4))[:, :, z]; g_et = (gt == 4)[:, :, z]
    for mask, c in [(g_wt, "yellow"), (g_tc, "cyan"), (g_et, "magenta")]:
        if mask.any(): axes[0].contour(mask.T, levels=[0.5], colors=[c], linewidths=0.8)
    axes[0].set_title("GT"); axes[0].axis("off")
    for ax, (title, pred_path) in zip(axes[1:], panels):
        ax.imshow(flair[:, :, z].T, cmap="gray", origin="lower")
        if pred_path.exists():
            p = nib.load(str(pred_path)).get_fdata().astype(np.int16)
            p_wt = (p > 0)[:, :, z]; p_tc = ((p == 1) | (p == 4))[:, :, z]; p_et = (p == 4)[:, :, z]
            for mask, c in [(p_wt, "yellow"), (p_tc, "cyan"), (p_et, "magenta")]:
                if mask.any(): ax.contour(mask.T, levels=[0.5], colors=[c], linewidths=0.8)
        ax.set_title(title, fontsize=9); ax.axis("off")
    fig.suptitle(f"{pid}  (z={z}, σ={sigma_chosen}) | WT=yellow TC=cyan ET=magenta", fontsize=10)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=140, bbox_inches="tight")
    plt.show()


# Pick the two test patients with the median and 75th-percentile tumor volumes
tumor_vols = []
for _, row in test_df.iterrows():
    seg = nib.load(row["seg"]).get_fdata()
    tumor_vols.append((row["pid"], (seg > 0).sum()))
tumor_vols.sort(key=lambda t: t[1])
pid_median = tumor_vols[len(tumor_vols) // 2][0]
pid_p75    = tumor_vols[3 * len(tumor_vols) // 4][0]
log.info("Rendering cases pid_median=%s pid_p75=%s", pid_median, pid_p75)

render_test_case(pid_median, CFG.out_root / "figures" / f"qualitative_{pid_median}.png")
render_test_case(pid_p75,    CFG.out_root / "figures" / f"qualitative_{pid_p75}.png")


In [ ]:
def headline_dsc(cond: str, region: str) -> Tuple[float, float]:
    sub = test_agg[(test_agg["condition"] == cond) & (test_agg["region"] == region)].iloc[0]
    return float(sub["DSC_mean"]), float(sub["DSC_std"])

print(f"σ_chosen = {sigma_chosen}")
print(f"Test subset: n = {len(test_df)} patients\n")
print("Headline WT DSC (mean ± std):")
for cond in conditions_order:
    m, s = headline_dsc(cond, "WT")
    print(f"  {cond:30s}  {m:.3f} ± {s:.3f}")
print("\nHeadline ET DSC (mean ± std):")
for cond in conditions_order:
    m, s = headline_dsc(cond, "ET")
    print(f"  {cond:30s}  {m:.3f} ± {s:.3f}")
print("\nWilcoxon p-values (WT DSC, two-sided):")
for r in wcx_rows:
    print(f"  {r['comparison']:55s}  raw p = {r['p_value_two_sided']:.4g}; Holm p = {r['p_value_holm']:.4g}  "
          f"(median Δ = {r['median_diff_A_minus_B']:+.4f})")


#### Interpretation

Interpret the newly generated tables, paired effect-size confidence intervals and adjusted
p-values. Do not assume the historical winner is reproduced. A nonsignificant comparison
does not establish equivalence. These synthetic settings do not establish clinical utility.


## Study 2: Multiplicative bias field


# Study 2: positive multiplicative bias-field corruption

This study reuses the Gaussian patient split and compares the same filter bank, N4 correction,
and an independently bias-augmented fine-tuned model. Bias strength is a **log-amplitude**:
`exp(strength * smooth_field)`. Fields remain positive and tissue is never clipped away.
This differs from the historical simulator; the original bias results require a new run.


## 1. Setup — imports, config, seeds

We import everything we need plus `SimpleITK` (for N4) and `scipy.ndimage` (for fast non-zero brain-mask operations during bias-field generation). The config dataclass mirrors the Gaussian study but adds a `bias_strengths` sweep grid and a separate `out_root` so the two studies don't overwrite each other.

In [ ]:
from __future__ import annotations

import json, logging, random, time, zlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import nibabel as nib
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.stats import wilcoxon
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import SimpleITK as sitk

import monai
from monai.bundle import ConfigParser
from monai.inferers import sliding_window_inference
from monai.metrics import compute_hausdorff_distance


print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("monai", monai.__version__, "| SimpleITK", sitk.Version_VersionString())


In [ ]:
import os
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path.cwd()


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print("Repository root:", REPO_ROOT)
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from brats_robustness.data import (
    build_manifest, make_split, load_or_create_split,
    load_4mod_volume as _load_volume, znorm_nonzero,
)
from brats_robustness.corruptions import (
    corrupt_gaussian_noise, gen_bias_field, corrupt_bias_field, make_filter_bank,
)
from brats_robustness.regions import probs_to_label_map, gt_to_regions, pred_to_regions
from brats_robustness.metrics import case_metrics, evaluate_condition, aggregate
from brats_robustness.inference import infer_volume as _infer_volume
from brats_robustness.training import make_region_loss
from brats_robustness.statistics import paired_comparisons, paired_vectors
from brats_robustness.reproducibility import experiment_identity, guard_run, guard_bundle, guard_prediction_dir

from brats_robustness.selection import select_severity

from brats_robustness.models import ensure_bundle


### 1.2 Config - separate output tree for bias-field experiment

In [ ]:
import os

@dataclass
class FineTuneConfig:
    n_steps: int = 3000
    log_every: int = 50
    val_every: int = 250
    batch_size: int = 2
    patch_size: Tuple[int, int, int] = (192, 192, 128)
    lr: float = 1e-5
    weight_decay: float = 1e-5
    bias_prob: float = 0.7
    bias_strength_min: float = 0.2
    num_workers: int = 0


@dataclass
class Config:
    # Set BRATS_DATA_ROOT to the folder containing MICCAI_BraTS2020_TrainingData
    # (see README → Data). The Gaussian notebook must be run first — this study
    # reuses its split, bundle, and results (BRATS_GAUSSIAN_OUT_ROOT).
    train_root: Path = Path(
        os.environ.get(
            "BRATS_DATA_ROOT",
            "./BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData",
        )
    )
    # Independent output tree from the Gaussian study.
    out_root: Path = Path(os.environ.get("BRATS_OUT_ROOT_BIAS", "./outputs/brats_robustness_bias_v3"))
    # Reuse the cached Gaussian-noise results for the recap/comparison sections.
    gaussian_out_root: Path = Path(
        os.environ.get("BRATS_GAUSSIAN_OUT_ROOT", "./outputs/brats_robustness_v3")
    )
    bundle_name: str = "brats_mri_segmentation"
    bundle_version: str = "0.5.3"
    bundle_revision: str = "6ea5b85f04151c71e5599a6f31000e5b042e8d80"
    bundle_dir: Path = Path(os.environ.get("BRATS_BUNDLE_DIR", "./monai_bundles/0.5.3"))

    seed: int = 1337
    device: str = "cuda:0"
    train_frac: float = 0.70
    val_frac:   float = 0.15

    roi_size: Tuple[int, int, int] = (240, 240, 160)
    sw_batch: int = 1
    sw_overlap: float = 0.5
    suffix_to_channel: Dict[str, int] = field(default_factory=lambda: {
        "t1ce": 0, "t1": 1, "t2": 2, "flair": 3,
    })

    # Bias-field sweep grid (analog of sigma_rel grid in the Gaussian notebook)
    sweep_strengths: Tuple[float, ...] = (0.2, 0.4, 0.6, 0.8, 1.0, 1.2)
    sweep_n_val: int = 12

    # Filter bank for bias-field — same 4 linear filters PLUS N4 bias correction
    filter_names: Tuple[str, ...] = (
        "none", "mean_3", "gaussian_0p5", "gaussian_1p0", "gaussian_1p5", "n4",
    )

    fine_tune: FineTuneConfig = field(default_factory=FineTuneConfig)

CFG = Config()
if CFG.fine_tune.num_workers != 0:
    raise ValueError("Use num_workers=0 for deterministic dataset RNG in this notebook")

if not CFG.train_root.exists():
    raise FileNotFoundError(
        f"BraTS training data not found at {CFG.train_root}.\n"
        "Set the BRATS_DATA_ROOT environment variable (see README → Data)."
    )
if not (CFG.gaussian_out_root / "manifests" / f"split_seed{CFG.seed}.csv").exists():
    raise FileNotFoundError(
        f"Gaussian-study split not found under {CFG.gaussian_out_root}.\n"
        "Run notebooks/Study 1 (Gaussian noise) first (see README → Running it)."
    )

guard_run(CFG.out_root, experiment_identity(CFG, REPO_ROOT, "bias"))

for sub in [
    "manifests", "predictions", "metrics", "figures", "checkpoints",
    "predictions/sweep", "predictions/test_final",
]:
    (CFG.out_root / sub).mkdir(parents=True, exist_ok=True)
print("Output root:", CFG.out_root)
print("Reusing Gaussian outputs from:", CFG.gaussian_out_root)


### 1.3 Logger + seeding

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("brats_bias")

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)
device = torch.device(CFG.device if torch.cuda.is_available() else "cpu")
log.info("seed=%d device=%s", CFG.seed, device)


## Shared patient split

Reuse Gaussian ID-only assignments and merge with the current manifest. Reject missing, additional or duplicate IDs. Counts depend on the available dataset and any audited pretraining exclusions.


In [ ]:
SUFFIXES_REQUIRED = ["flair", "t1", "t1ce", "t2", "seg"]


manifest = build_manifest(CFG.train_root)
SPLIT_CSV = CFG.gaussian_out_root / "manifests" / f"split_seed{CFG.seed}.csv"
assert SPLIT_CSV.exists(), f"Expected persisted Gaussian split at {SPLIT_CSV}."
split_assignments = pd.read_csv(SPLIT_CSV, usecols=["pid", "split"])
assert set(split_assignments.pid) == set(manifest.pid), "Gaussian split does not match current dataset IDs"
split_df = split_assignments.merge(manifest, on="pid", how="left", validate="one_to_one")
assert not split_df[["flair", "t1", "t1ce", "t2", "seg"]].isna().any().any(), "Persisted split contains patients missing from the current manifest."
train_df = split_df[split_df["split"] == "train"].reset_index(drop=True)
val_df = split_df[split_df["split"] == "val"].reset_index(drop=True)
test_df = split_df[split_df["split"] == "test"].reset_index(drop=True)
assert len(train_df) + len(val_df) + len(test_df) == len(split_df)
log.info("Loaded persisted split: train=%d val=%d test=%d", len(train_df), len(val_df), len(test_df))

## 3. Bundle + inference helpers (same as Gaussian study)

Brief — the bundle is already downloaded and patched. We load it and define `load_4mod_volume`, `znorm_nonzero`, `infer_volume`, `probs_to_label_map`, `save_nifti` exactly as in `Study 1 (Gaussian noise)`.

In [ ]:
bundle_path = CFG.bundle_dir / CFG.bundle_name
assert (bundle_path / "models" / "model.pt").exists(), \
    f"Bundle not found at {bundle_path}. Run the Gaussian notebook first to download it."

inference_cfg_path = bundle_path / "configs" / "inference.json"
# Bundle's inference.json was already patched in the Gaussian run; re-patch idempotently.
cfg_json = json.loads(inference_cfg_path.read_text())
if cfg_json.get("network_def", {}).get("_target_") == "SegResNet":
    cfg_json["network_def"]["_target_"] = "monai.networks.nets.SegResNet"
    inference_cfg_path.write_text(json.dumps(cfg_json, indent=4))

metadata = json.loads((bundle_path / "configs" / "metadata.json").read_text())
declared = metadata["network_data_format"]["inputs"]["image"]["channel_def"]
log.info("Bundle input channel_def: %s", declared)
# Sanity check the channel mapping matches what we have.
def _norm(s):
    s = s.lower().replace("-", "").replace(" ", "")
    return "flair" if "flair" in s else ("t1ce" if ("t1c" in s or "t1gd" in s) else ("t1" if (s=="t1" or s.startswith("t1w")) else "t2"))
declared_norm = {int(k): _norm(v) for k, v in declared.items()}
expected_norm = {idx: suf for suf, idx in CFG.suffix_to_channel.items()}
assert declared_norm == expected_norm, (declared_norm, expected_norm)
log.info("✓ channel order verified — FLAIR is channel %d", CFG.suffix_to_channel["flair"])

parser = ConfigParser(); parser.read_config(str(inference_cfg_path))
network = parser.get_parsed_content("network_def").to(device)
state = torch.load(bundle_path / "models" / "model.pt", map_location=device, weights_only=False)
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]
network.load_state_dict(state, strict=True)
network.eval()
log.info("Pretrained network loaded.")

guard_bundle(CFG.out_root, bundle_path, CFG.bundle_version)


In [ ]:
def load_4mod_volume(row):
    return _load_volume(row, CFG.suffix_to_channel)


def infer_volume(net, vol):
    return _infer_volume(net, vol, CFG, device)


def save_nifti(arr, affine, out_path):
    # Write in the same directory and atomically publish a complete prediction.
    out_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = out_path.with_name(out_path.name.replace(".nii.gz", ".partial.nii.gz"))
    nib.save(nib.Nifti1Image(arr, affine), str(temporary))
    temporary.replace(out_path)


def patient_noise_seed(pid):
    # Full CRC32 avoids the original modulo-10000 seed collisions.
    return CFG.seed + zlib.crc32(pid.encode())


## 4. Recap — what we found with Gaussian noise

Before introducing the new corruption, a one-page summary of the Gaussian-noise study (`Study 1 (Gaussian noise)`):

In [ ]:
sigma_chosen = float((CFG.gaussian_out_root / "metrics" / "sigma_chosen.txt").read_text())
# Load the cached Gaussian results so we can show them inline.
gauss_agg = pd.read_csv(CFG.gaussian_out_root / "metrics" / "test_aggregate.csv")
gauss_wcx = pd.read_csv(CFG.gaussian_out_root / "metrics" / "test_wilcoxon_WT_DSC.csv")

print("=== Gaussian-noise study: test WT-DSC by condition (mean ± std) ===")
gauss_wt = gauss_agg[gauss_agg.region == "WT"][["condition", "DSC_mean", "DSC_std"]]
gauss_wt = gauss_wt.sort_values("DSC_mean", ascending=False).reset_index(drop=True)
print(gauss_wt.to_string(index=False, formatters={"DSC_mean":"{:.3f}".format, "DSC_std":"{:.3f}".format}))
print()
print("=== Gaussian-noise study: Wilcoxon (paired, two-sided, on WT-DSC) ===")
print(gauss_wcx[["comparison", "median_diff_A_minus_B", "p_value_two_sided"]].to_string(
    index=False, formatters={"median_diff_A_minus_B":"{:+.4f}".format,
                              "p_value_two_sided":"{:.2e}".format}))


## Positive bias-field simulator

A deterministic smooth field is built from Gaussian blobs and normalized to maximum absolute
amplitude 1, then exponentiated. At strength s its multiplier lies between exp(-s) and exp(s).
This is a synthetic intensity-inhomogeneity proxy, not a calibrated scanner-artifact model.
Noise severity and bias log-amplitude are different scales and cannot be compared directly.


In [ ]:
# Positive log-amplitude bias simulator is shared in brats_robustness/corruptions.py.
# Its severity is different from the historical clipped simulator.


### 5.1 Visualize the bias-field corruption

Three subplots for one patient: clean FLAIR, corrupted FLAIR at strength=1.0, and the bias field itself. This makes the *structural* nature of the corruption obvious — it's not noise, it's a slow intensity warp.

In [ ]:
demo_pid = val_df.iloc[0]["pid"]
demo_flair = nib.load(val_df.iloc[0]["flair"]).get_fdata().astype(np.float32)
demo_seg   = nib.load(val_df.iloc[0]["seg"]  ).get_fdata().astype(np.int16)
z = int(np.argmax((demo_seg > 0).sum(axis=(0, 1))))

field = gen_bias_field(demo_flair.shape, seed=patient_noise_seed(demo_pid), strength=1.0)
corrupted = corrupt_bias_field(demo_flair, 1.0, seed=patient_noise_seed(demo_pid))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(demo_flair[:, :, z].T, cmap="gray", origin="lower")
axes[0].set_title(f"Clean FLAIR ({demo_pid}, z={z})")
axes[0].axis("off")
axes[1].imshow(corrupted[:, :, z].T, cmap="gray", origin="lower")
axes[1].set_title("Corrupted FLAIR — bias field, strength = 1.0")
axes[1].axis("off")
im = axes[2].imshow(field[:, :, z].T, cmap="coolwarm", origin="lower", vmin=float(field.min()), vmax=float(field.max()))
axes[2].set_title("Bias field (multiplicative)")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "bias_field_demo.png", dpi=140, bbox_inches="tight")
plt.show()


## Preprocessing comparison

Compare the same fixed smoothing filters with N4 bias-field correction. Every method receives the clean brain support mask. N4 uses three resolution levels with 20 iterations per level and explicit isotropic 1mm spacing; runtime and recovery must be measured on the rerun.


In [ ]:
FILTER_BANK = make_filter_bank(include_n4=True)
log.info("Filter bank: %s", list(FILTER_BANK))


## 7. Metrics + orchestrators (same machinery as Gaussian study)

Per-region (WT / TC / ET) DSC, HD95 (mm), sensitivity, FPvol (mL), FNvol (mL); aggregator with mean ± std and median [IQR]; idempotent inference orchestrators that cache predictions to NIfTI.

In [ ]:
def infer_subset_clean(net, df_subset, out_dir, desc):
    guard_prediction_dir(out_dir, net, {"kind": "clean"})
    for _, row in tqdm(list(df_subset.iterrows()), desc=desc):
        pid = row["pid"]
        out_path = out_dir / f"{pid}_pred.nii.gz"
        if out_path.exists():
            continue
        vol, affine = load_4mod_volume(row)
        probs = infer_volume(net, vol)
        save_nifti(probs_to_label_map(probs), affine, out_path)


def infer_subset_biased(net, df_subset, out_dir, desc, strength, filter_fn):
    """Corrupt FLAIR with bias field at `strength` (per-patient seed), apply filter_fn, infer."""
    guard_prediction_dir(out_dir, net, {"kind": "bias", "severity": float(strength), "preprocessing": filter_fn.preprocessing_name})
    flair_idx = CFG.suffix_to_channel["flair"]
    for _, row in tqdm(list(df_subset.iterrows()), desc=desc):
        pid = row["pid"]
        out_path = out_dir / f"{pid}_pred.nii.gz"
        if out_path.exists():
            continue
        vol, affine = load_4mod_volume(row)
        biased = corrupt_bias_field(vol[flair_idx], strength, seed=patient_noise_seed(pid))
        brain_mask = vol[flair_idx] != 0
        vol[flair_idx] = filter_fn(biased, brain_mask)
        probs = infer_volume(net, vol)
        save_nifti(probs_to_label_map(probs), affine, out_path)


## Bias validation sweep

Sweep log-amplitude levels 0.2 through 1.2. Use the same unmitigated validation-drop selection rule as Gaussian noise. The axes still represent different corruption mechanisms and are not calibrated physical severity scales. Select the best linear filter using validation scores only.


In [ ]:
SWEEP_VAL_DF = val_df.iloc[:CFG.sweep_n_val].reset_index(drop=True)
SWEEP_DIR = CFG.out_root / "predictions" / "sweep"
CLEAN_VAL_DIR = SWEEP_DIR / "clean"
log.info("Sweep on %d val patients, strengths=%s", len(SWEEP_VAL_DF), CFG.sweep_strengths)

# Clean inference once for the sweep val subset
infer_subset_clean(network, SWEEP_VAL_DF, CLEAN_VAL_DIR, "sweep clean")

# Run each (strength, filter) combination
for s in CFG.sweep_strengths:
    s_tag = f"strength_{s:.2f}".replace(".", "p")
    for fname in CFG.filter_names:
        out_dir = SWEEP_DIR / s_tag / fname
        infer_subset_biased(
            net=network, df_subset=SWEEP_VAL_DF,
            out_dir=out_dir, desc=f"sweep {s_tag}/{fname}",
            strength=s, filter_fn=FILTER_BANK[fname],
        )
log.info("Sweep inference done.")


### Select bias severity

Use the unmitigated corruption drop on validation patients, independently of N4 or fine-tuning performance. Record the decision and fallback status.


In [ ]:
sweep_rows = []
clean_eval = evaluate_condition(SWEEP_VAL_DF, CLEAN_VAL_DIR, "Clean")
clean_eval["strength"] = np.nan; clean_eval["filter"] = "clean"
sweep_rows.append(clean_eval)

for s in CFG.sweep_strengths:
    s_tag = f"strength_{s:.2f}".replace(".", "p")
    for fname in CFG.filter_names:
        out_dir = SWEEP_DIR / s_tag / fname
        e = evaluate_condition(SWEEP_VAL_DF, out_dir, f"s{s}_{fname}")
        e["strength"] = s; e["filter"] = fname
        sweep_rows.append(e)

sweep_df = pd.concat(sweep_rows, ignore_index=True)
sweep_df.to_csv(CFG.out_root / "metrics" / "sweep_per_case.csv", index=False)

wt = sweep_df[sweep_df.region == "WT"]
clean_wt = wt[wt["filter"] == "clean"]["DSC"].mean()
log.info("Sweep clean WT-DSC mean = %.4f", clean_wt)

summary = []
for s in CFG.sweep_strengths:
    for fname in CFG.filter_names:
        sel = wt[(wt["filter"] == fname) & (wt.strength == s)]
        if not sel.empty:
            summary.append({"strength": s, "filter": fname,
                            "DSC_WT_mean": sel.DSC.mean(),
                            "DSC_WT_std":  sel.DSC.std()})
sweep_summary = pd.DataFrame(summary)
sweep_summary.to_csv(CFG.out_root / "metrics" / "sweep_summary.csv", index=False)
print(sweep_summary.pivot(index="strength", columns="filter", values="DSC_WT_mean").round(3))


In [ ]:
corrupted_validation_means = [
    float(wt[(wt["filter"] == "none") & (wt["strength"] == s)].DSC.mean())
    for s in CFG.sweep_strengths
]
strength_chosen, severity_reason = select_severity(
    CFG.sweep_strengths, clean_wt, corrupted_validation_means,
)
(CFG.out_root / "metrics" / "strength_chosen.txt").write_text(str(strength_chosen))
(CFG.out_root / "metrics" / "severity_selection.json").write_text(
    json.dumps({"chosen": strength_chosen, "reason": severity_reason, "target_drop": 0.08}, indent=2)
)
log.info("Chosen bias log-amplitude=%s: %s", strength_chosen, severity_reason)


### 8.2 Plot DSC vs bias-field strength

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(clean_wt, color="gray", ls="--", label=f"clean ({clean_wt:.3f})")
markers = {"none":"o", "mean_3":"s", "gaussian_0p5":"v", "gaussian_1p0":"D", "gaussian_1p5":"P", "n4":"*"}
colors  = {"none":"#d62728", "mean_3":"#17becf", "gaussian_0p5":"#ff9896",
           "gaussian_1p0":"#2ca02c", "gaussian_1p5":"#98df8a", "n4":"#9467bd"}
for fname in CFG.filter_names:
    sub = sweep_summary[sweep_summary["filter"] == fname].sort_values("strength")
    ax.errorbar(sub.strength, sub.DSC_WT_mean, yerr=sub.DSC_WT_std,
                marker=markers[fname], color=colors[fname], capsize=3,
                label=fname, linewidth=1.5,
                markersize=8 if fname == "n4" else 6)
ax.axvline(strength_chosen, color="red", ls=":", alpha=0.6,
           label=f"chosen strength={strength_chosen}")
ax.set_xlabel("Bias-field strength")
ax.set_ylabel("mean WT DSC")
ax.set_ylim(0, 1)
ax.set_title(f"Bias-field sweep on val (n={len(SWEEP_VAL_DF)}): each filter vs strength")
ax.grid(alpha=0.3); ax.legend(loc="lower left", fontsize=9)
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "sweep_DSC_WT_vs_strength.png", dpi=140, bbox_inches="tight")
plt.show()

# Selection timing: The linear filter is selected from validation results before final test inference.


## 9. Bias-augmented fine-tuning

Analog of the Gaussian fine-tune, but the FLAIR augmentation is a *bias field* with random strength `~ Uniform(0.2, strength_chosen)` (with probability 0.7 — 30 % of samples kept clean to guard against forgetting). Same optimizer, same LR, same number of steps. The val-DSC checkpoint selection uses bias-corrupted val patients.

In [ ]:
class BiasPatchDataset(torch.utils.data.Dataset):
    """Random brain-centered patches with bias-field FLAIR augmentation."""

    def __init__(self, df, patch_size, bias_prob, strength_max, random_state):
        self.df = df.reset_index(drop=True)
        self.patch = patch_size
        self.bias_prob = bias_prob
        self.s_lo, self.s_hi = CFG.fine_tune.bias_strength_min, strength_max
        self.rng = np.random.default_rng(random_state)

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _seg_to_regions(seg):
        return np.stack([
            (seg == 1) | (seg == 4),
            seg > 0,
            seg == 4,
        ], axis=0).astype(np.float32)

    def _random_patch(self, vol, gt_r):
        H, W, D = vol.shape[1:]
        ph, pw, pd_ = self.patch
        brain = vol[CFG.suffix_to_channel["flair"]] != 0
        if brain.any():
            zs = np.argwhere(brain)
            c = zs[self.rng.integers(0, len(zs))]
            h0 = int(np.clip(c[0] - ph // 2, 0, H - ph))
            w0 = int(np.clip(c[1] - pw // 2, 0, W - pw))
            d0 = int(np.clip(c[2] - pd_ // 2, 0, D - pd_))
        else:
            h0 = int(self.rng.integers(0, max(1, H - ph + 1)))
            w0 = int(self.rng.integers(0, max(1, W - pw + 1)))
            d0 = int(self.rng.integers(0, max(1, D - pd_ + 1)))
        return vol[:, h0:h0+ph, w0:w0+pw, d0:d0+pd_], gt_r[:, h0:h0+ph, w0:w0+pw, d0:d0+pd_]

    def __getitem__(self, i):
        row = self.df.iloc[i]
        vol, _ = load_4mod_volume(row)
        seg = nib.load(row["seg"]).get_fdata().astype(np.int16)
        if self.rng.random() < self.bias_prob:
            s = float(self.rng.uniform(self.s_lo, self.s_hi))
            aug_seed = int(self.rng.integers(0, 2**31 - 1))
            vol[CFG.suffix_to_channel["flair"]] = corrupt_bias_field(
                vol[CFG.suffix_to_channel["flair"]], s, seed=aug_seed
            )
        vol = znorm_nonzero(vol)
        gt_r = self._seg_to_regions(seg)
        v, g = self._random_patch(vol, gt_r)
        return torch.from_numpy(v).float(), torch.from_numpy(g).float()


### 9.1 Training loop

In [ ]:
def _quick_val_dsc(net, df_val_subset, strength):
    """Mean WT-DSC under bias-field corruption (no filter) — used as the val signal."""
    net.eval()
    flair_idx = CFG.suffix_to_channel["flair"]
    dscs = []
    with torch.no_grad():
        for _, row in df_val_subset.iterrows():
            vol, _ = load_4mod_volume(row)
            vol[flair_idx] = corrupt_bias_field(
                vol[flair_idx], strength, seed=patient_noise_seed(row["pid"])
            )
            probs = infer_volume(net, vol)
            seg = nib.load(row["seg"]).get_fdata().astype(np.int16)
            p_wt = probs_to_label_map(probs) > 0; g_wt = seg > 0
            inter = np.logical_and(p_wt, g_wt).sum()
            denom = p_wt.sum() + g_wt.sum()
            dscs.append(2 * inter / denom if denom > 0 else 1.0)
    return float(np.mean(dscs))


# Idempotent: rebuild network from bundle weights at each (re)run of this cell.
parser_ft = ConfigParser(); parser_ft.read_config(str(inference_cfg_path))
net_ft = parser_ft.get_parsed_content("network_def").to(device)
state_ft = torch.load(bundle_path / "models" / "model.pt", map_location=device, weights_only=False)
if isinstance(state_ft, dict) and "state_dict" in state_ft:
    state_ft = state_ft["state_dict"]
net_ft.load_state_dict(state_ft, strict=True)

CHECKPOINT_BEST = CFG.out_root / "checkpoints" / "best_finetuned.pt"
LOSS_HISTORY_NPZ = CFG.out_root / "checkpoints" / "loss_history.npz"

ft_cfg = CFG.fine_tune
train_ds = BiasPatchDataset(
    train_df, patch_size=ft_cfg.patch_size,
    bias_prob=ft_cfg.bias_prob, strength_max=float(strength_chosen),
    random_state=CFG.seed,
)
loader = torch.utils.data.DataLoader(
    train_ds, batch_size=ft_cfg.batch_size, shuffle=True,
    num_workers=ft_cfg.num_workers, pin_memory=True, drop_last=True,
    generator=torch.Generator().manual_seed(CFG.seed),
)
loader_iter = iter(loader)

optimizer = torch.optim.AdamW(net_ft.parameters(), lr=ft_cfg.lr, weight_decay=ft_cfg.weight_decay)
loss_fn = make_region_loss()
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

val_subset_for_ft = val_df.reset_index(drop=True)

if CHECKPOINT_BEST.exists():
    log.info("Cached fine-tune checkpoint found — skipping training. Delete %s to retrain.", CHECKPOINT_BEST)
    if LOSS_HISTORY_NPZ.exists():
        history = np.load(LOSS_HISTORY_NPZ, allow_pickle=True)
        losses = history["losses"].tolist()
        val_dsc_history = history["val_dsc_history"].tolist()
    else:
        log.warning("loss_history.npz missing — synthesizing empty history for plotting")
        losses, val_dsc_history = [], []
else:
    net_ft.train()
    losses, val_dsc_history = [], []
    best_val = -1.0
    pbar = tqdm(range(1, ft_cfg.n_steps + 1), desc="fine-tune (bias)")
    for step in pbar:
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader); x, y = next(loader_iter)
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            logits = net_ft(x); loss = loss_fn(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        losses.append(float(loss.item()))
        if step % ft_cfg.log_every == 0:
            pbar.set_postfix(loss=f"{np.mean(losses[-ft_cfg.log_every:]):.4f}")
        if step % ft_cfg.val_every == 0 or step == ft_cfg.n_steps:
            v = _quick_val_dsc(net_ft, val_subset_for_ft, strength=float(strength_chosen))
            val_dsc_history.append((step, v))
            log.info("step %d  val WT-DSC (s=%s) = %.4f  best=%.4f", step, strength_chosen, v, best_val)
            if v > best_val:
                best_val = v
                torch.save(net_ft.state_dict(), CHECKPOINT_BEST)
                log.info("  → new best, saved")
            # Persist history every val tick so a crash doesn't wipe it.
            np.savez(LOSS_HISTORY_NPZ,
                     losses=np.array(losses),
                     val_dsc_history=np.array(val_dsc_history, dtype=object))
            net_ft.train()
    np.savez(LOSS_HISTORY_NPZ,
             losses=np.array(losses),
             val_dsc_history=np.array(val_dsc_history, dtype=object))

net_ft.load_state_dict(torch.load(CHECKPOINT_BEST, map_location=device, weights_only=False))
net_ft.eval()
log.info("Bias-augmented fine-tune done. Best checkpoint loaded.")


### 9.2 Loss + val-DSC curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
losses_arr = np.array(losses)
ax1.plot(losses_arr, color="lightgray", alpha=0.5, label="step loss")
if len(losses_arr) >= 50:
    k = 50
    running = np.convolve(losses_arr, np.ones(k)/k, mode="valid")
    ax1.plot(np.arange(k-1, len(losses_arr)), running, color="C0", label=f"running mean ({k})")
ax1.set_xlabel("step"); ax1.set_ylabel("DiceCE loss"); ax1.set_title("Bias-aug fine-tune loss")
ax1.grid(alpha=0.3); ax1.legend()

if val_dsc_history:
    steps_v = [int(s) for s, _ in val_dsc_history]
    vals_v  = [float(v) for _, v in val_dsc_history]
    ax2.plot(steps_v, vals_v, marker="o")
    ax2.set_xlabel("step"); ax2.set_ylabel(f"val WT-DSC @ strength={strength_chosen}")
    ax2.set_title("Val WT-DSC under bias corruption")
    ax2.set_ylim(0, 1); ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "finetune_curves.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# Select the best linear filter from the bias validation sweep, never from test scores.
validation_linear_names = ["mean_3", "gaussian_0p5", "gaussian_1p0", "gaussian_1p5"]
bias_validation_filters = sweep_summary[
    (sweep_summary["strength"] == strength_chosen)
    & (sweep_summary["filter"].isin(validation_linear_names))
]
assert not bias_validation_filters.empty, "No bias validation filter scores found for strength_chosen."
best_linear_name = bias_validation_filters.loc[
    bias_validation_filters["DSC_WT_mean"].idxmax(), "filter"
]
best_linear = f"Filter {best_linear_name}"
(CFG.out_root / "metrics" / "selected_linear_filter.txt").write_text(best_linear)
log.info("Selected linear filter from validation: %s", best_linear)


## 10. Final test evaluation — bias-field family

Nine conditions on the local held-out test set, all at `strength = strength_chosen`:

1. Clean (pretrained)
2. Corrupted (pretrained), no filter
3. Filter mean_3
4. Filter gaussian_0p5
5. Filter gaussian_1p0
6. Filter gaussian_1p5
7. **Filter N4** (proper data-level fix)
8. Corrupted (fine-tuned)
9. Clean (fine-tuned) — forgetting check

In [ ]:
TEST_DIR = CFG.out_root / "predictions" / "test_final"
strength_chosen_f = float(strength_chosen)

# 1. Clean (pretrained)
infer_subset_clean(network, test_df, TEST_DIR / "clean_pretrained", "test: clean pretrained")

# 2. Corrupted (pretrained), no filter
infer_subset_biased(network, test_df, TEST_DIR / "corrupted_pretrained",
                    "test: corrupted pretrained", strength=strength_chosen_f,
                    filter_fn=FILTER_BANK["none"])

# 3–7. Filters
for fname in CFG.filter_names:
    if fname == "none":
        continue
    infer_subset_biased(network, test_df, TEST_DIR / f"filter_{fname}",
                        f"test: filter {fname}", strength=strength_chosen_f,
                        filter_fn=FILTER_BANK[fname])

# 8. Corrupted (fine-tuned)
infer_subset_biased(net_ft, test_df, TEST_DIR / "corrupted_finetuned",
                    "test: corrupted finetuned", strength=strength_chosen_f,
                    filter_fn=FILTER_BANK["none"])

# 9. Clean (fine-tuned) — forgetting check
infer_subset_clean(net_ft, test_df, TEST_DIR / "clean_finetuned", "test: clean finetuned")

log.info("All bias-field test predictions saved.")


### 10.1 Per-case metrics, aggregate, Wilcoxon

In [ ]:
COND_DIRS = {
    "Clean (pretrained)":      TEST_DIR / "clean_pretrained",
    "Corrupted (pretrained)":  TEST_DIR / "corrupted_pretrained",
    "Filter mean_3":           TEST_DIR / "filter_mean_3",
    "Filter gaussian_0p5":     TEST_DIR / "filter_gaussian_0p5",
    "Filter gaussian_1p0":     TEST_DIR / "filter_gaussian_1p0",
    "Filter gaussian_1p5":     TEST_DIR / "filter_gaussian_1p5",
    "Filter N4":               TEST_DIR / "filter_n4",
    "Corrupted (fine-tuned)":  TEST_DIR / "corrupted_finetuned",
    "Clean (fine-tuned)":      TEST_DIR / "clean_finetuned",
}

per_case_rows = []
for cond, d in COND_DIRS.items():
    per_case_rows.append(evaluate_condition(test_df, d, cond))
test_per_case = pd.concat(per_case_rows, ignore_index=True)
test_per_case.to_csv(CFG.out_root / "metrics" / "test_per_case.csv", index=False)

test_agg = aggregate(test_per_case)
test_agg.to_csv(CFG.out_root / "metrics" / "test_aggregate.csv", index=False)

print("\n=== Bias-field study: test DSC summary ===")
v = test_agg.pivot(index="condition", columns="region", values=["DSC_mean", "DSC_std"])
v = v.swaplevel(0, 1, axis=1).sort_index(axis=1)
print(v.round(3).to_string())
test_agg


In [ ]:
def wt_dsc_vec(df, cond):
    return df[(df.condition == cond) & (df.region == "WT")].sort_values("pid")["DSC"].to_numpy()


pairs = [
    ("Corrupted vs Clean (pretrained)",              "Corrupted (pretrained)", "Clean (pretrained)"),
    (f"Best linear filter ({best_linear}) vs Corrupted (pretrained)", best_linear, "Corrupted (pretrained)"),
    ("Filter N4 vs Corrupted (pretrained)",          "Filter N4",              "Corrupted (pretrained)"),
    ("Fine-tune vs Filter N4",                        "Corrupted (fine-tuned)", "Filter N4"),
    ("Fine-tune vs best linear filter",              "Corrupted (fine-tuned)", best_linear),
    ("Forgetting: Clean (fine-tuned) vs Clean (pretrained)", "Clean (fine-tuned)", "Clean (pretrained)"),
]

wcx_rows = paired_comparisons(test_per_case, pairs, expected_n=len(test_df), seed=CFG.seed)
wcx_df = pd.DataFrame(wcx_rows)
wcx_df.to_csv(CFG.out_root / "metrics" / "test_wilcoxon_WT_DSC.csv", index=False)
wcx_df


### 10.2 Bar chart — mean DSC per condition × region (bias-field family)

In [ ]:
regions = ["WT", "TC", "ET"]
conditions_order = list(COND_DIRS.keys())
fig, ax = plt.subplots(figsize=(15, 5))
xs = np.arange(len(conditions_order)); width = 0.27
for i, region in enumerate(regions):
    sub = test_agg[test_agg.region == region].set_index("condition")
    means = sub.loc[conditions_order, "DSC_mean"].values
    stds  = sub.loc[conditions_order, "DSC_std"].values
    ax.bar(xs + (i-1)*width, means, width, yerr=stds, label=region, capsize=2)
ax.set_xticks(xs); ax.set_xticklabels(conditions_order, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("DSC (mean ± std)"); ax.set_ylim(0, 1)
ax.set_title(f"Bias-field family — test DSC by condition × region (n={len(test_df)}, strength={strength_chosen})")
ax.grid(alpha=0.3, axis="y"); ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "test_dsc_bar_bias.png", dpi=140, bbox_inches="tight")
plt.show()


### 10.3 Qualitative comparison for one representative test case

In [ ]:
def render_case(pid, out_png):
    row = test_df[test_df.pid == pid].iloc[0]
    flair_orig = nib.load(row["flair"]).get_fdata().astype(np.float32)
    gt = nib.load(row["seg"]).get_fdata().astype(np.int16)
    z = int(np.argmax((gt > 0).sum(axis=(0, 1))))
    # Show clean FLAIR for backdrop (consistent with other notebook)
    panels = [
        ("Clean (pretrained)",      TEST_DIR / "clean_pretrained"    / f"{pid}_pred.nii.gz"),
        ("Corrupted (pretrained)",  TEST_DIR / "corrupted_pretrained" / f"{pid}_pred.nii.gz"),
        (best_linear,               COND_DIRS[best_linear]            / f"{pid}_pred.nii.gz"),
        ("Filter N4",               TEST_DIR / "filter_n4"            / f"{pid}_pred.nii.gz"),
        ("Corrupted (fine-tuned)",  TEST_DIR / "corrupted_finetuned"  / f"{pid}_pred.nii.gz"),
    ]
    fig, axes = plt.subplots(1, len(panels) + 1, figsize=(3.0 * (len(panels) + 1), 3.5))
    axes[0].imshow(flair_orig[:, :, z].T, cmap="gray", origin="lower")
    g_wt = (gt > 0)[:, :, z]; g_tc = ((gt == 1) | (gt == 4))[:, :, z]; g_et = (gt == 4)[:, :, z]
    for mask, c in [(g_wt, "yellow"), (g_tc, "cyan"), (g_et, "magenta")]:
        if mask.any():
            axes[0].contour(mask.T, levels=[0.5], colors=[c], linewidths=0.8)
    axes[0].set_title("GT"); axes[0].axis("off")
    for ax, (title, pred_path) in zip(axes[1:], panels):
        ax.imshow(flair_orig[:, :, z].T, cmap="gray", origin="lower")
        if pred_path.exists():
            p = nib.load(str(pred_path)).get_fdata().astype(np.int16)
            p_wt = (p > 0)[:, :, z]; p_tc = ((p == 1) | (p == 4))[:, :, z]; p_et = (p == 4)[:, :, z]
            for mask, c in [(p_wt, "yellow"), (p_tc, "cyan"), (p_et, "magenta")]:
                if mask.any():
                    ax.contour(mask.T, levels=[0.5], colors=[c], linewidths=0.8)
        ax.set_title(title, fontsize=9); ax.axis("off")
    fig.suptitle(f"{pid}  (z={z}, bias strength={strength_chosen}) | WT=yellow TC=cyan ET=magenta",
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(out_png, dpi=140, bbox_inches="tight")
    plt.show()


tumor_vols = []
for _, row in test_df.iterrows():
    seg = nib.load(row["seg"]).get_fdata()
    tumor_vols.append((row["pid"], (seg > 0).sum()))
tumor_vols.sort(key=lambda t: t[1])
pid_median = tumor_vols[len(tumor_vols) // 2][0]
log.info("Rendering median-tumor case: %s", pid_median)
render_case(pid_median, CFG.out_root / "figures" / f"qualitative_{pid_median}.png")


## 11. Side-by-side comparison of the two corruption families

We now have full results for both corruptions on the *same* test patients. The contrast between them is the headline of this companion study.

In [ ]:
bias_agg  = test_agg.copy()
gauss_agg = pd.read_csv(CFG.gaussian_out_root / "metrics" / "test_aggregate.csv")

# Restrict to WT for the headline comparison
bias_wt  = bias_agg [bias_agg .region == "WT"].set_index("condition")
gauss_wt = gauss_agg[gauss_agg.region == "WT"].set_index("condition")

print(f"=== Gaussian noise (σ={sigma_chosen}) — WT-DSC (mean) ===")
print(gauss_wt[["DSC_mean", "DSC_std"]].sort_values("DSC_mean", ascending=False).round(3))
print()
print(f"=== Bias field (strength={strength_chosen}) — WT-DSC (mean) ===")
print(bias_wt [["DSC_mean", "DSC_std"]].sort_values("DSC_mean", ascending=False).round(3))


### 11.1 Side-by-side bar chart — Gaussian vs bias for the headline conditions

In [ ]:
# Map condition → a stable label so the two studies line up.
LABEL_MAP = {
    "Clean (pretrained)":         "Clean (pretrained)",
    "Corrupted (pretrained)":     "Corrupted",
    "Filter mean_3":              "Mean 3×3×3 filter",
    "Filter gaussian_1p0":        "Gaussian σ=1 filter",
    "Filter N4":                  "N4 bias correction",
    "Corrupted (fine-tuned)":     "Fine-tuned",
    "Clean (fine-tuned)":         "Clean (fine-tuned)",
}

# For Gaussian, N4 doesn't exist → use NaN placeholder; for bias, N4 is the headline.
def get_wt_mean(agg_wt, cond):
    if cond not in agg_wt.index:
        return np.nan, np.nan
    return float(agg_wt.loc[cond, "DSC_mean"]), float(agg_wt.loc[cond, "DSC_std"])

display_conditions = [
    "Clean (pretrained)", "Corrupted (pretrained)",
    "Filter mean_3", "Filter gaussian_1p0", "Filter N4",
    "Corrupted (fine-tuned)", "Clean (fine-tuned)",
]
gauss_means = [get_wt_mean(gauss_wt, c)[0] for c in display_conditions]
gauss_stds  = [get_wt_mean(gauss_wt, c)[1] for c in display_conditions]
bias_means  = [get_wt_mean(bias_wt,  c)[0] for c in display_conditions]
bias_stds   = [get_wt_mean(bias_wt,  c)[1] for c in display_conditions]

fig, ax = plt.subplots(figsize=(13, 5.5))
xs = np.arange(len(display_conditions))
W = 0.4
gauss_bars = ax.bar(xs - W/2, gauss_means, W, yerr=gauss_stds, color="#d62728", alpha=0.85,
       label=f"Gaussian noise σ={sigma_chosen}", edgecolor="black", linewidth=0.5, capsize=3)
bias_bars  = ax.bar(xs + W/2, bias_means,  W, yerr=bias_stds, color="#9467bd", alpha=0.85,
       label=f"Bias field strength={strength_chosen}", edgecolor="black", linewidth=0.5, capsize=3)
for b, m in zip(gauss_bars, gauss_means):
    if not np.isnan(m):
        ax.text(b.get_x() + b.get_width()/2, m + 0.005, f"{m:.3f}", ha="center", va="bottom", fontsize=8)
for b, m in zip(bias_bars, bias_means):
    if not np.isnan(m):
        ax.text(b.get_x() + b.get_width()/2, m + 0.005, f"{m:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels([LABEL_MAP[c] for c in display_conditions], rotation=20, ha="right")
ax.set_ylabel("Whole-Tumor DSC (mean ± std)")
ax.set_ylim(0.5, 1.0); ax.grid(alpha=0.3, axis="y")
ax.legend(loc="lower left", fontsize=11)
ax.set_title("Side-by-side: which mitigation wins per corruption family (test set, n=55)")
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "comparison_gauss_vs_bias.png", dpi=160, bbox_inches="tight")
plt.show()


### 11.2 Lift-over-corrupted plot — quantifies each mitigation

In [ ]:
mitigations_gauss = ["Filter mean_3", "Filter gaussian_1p0", "Corrupted (fine-tuned)"]
mitigations_bias  = ["Filter mean_3", "Filter gaussian_1p0", "Filter N4", "Corrupted (fine-tuned)"]

def lift(per_case, cond, baseline_cond):
    a, b = paired_vectors(per_case, cond, baseline_cond)
    return float(np.mean(a - b)), float(np.std(a - b))

def wt_dsc_vec_pc(per_case, cond):
    return per_case[(per_case.condition == cond) & (per_case.region == "WT")].sort_values("pid")["DSC"].to_numpy()

gauss_pc = pd.read_csv(CFG.gaussian_out_root / "metrics" / "test_per_case.csv")
bias_pc  = test_per_case

# Build lift dict: condition → (mean, std) of (mitigation − corrupted) per patient
def collect_lifts(per_case, mitigations, baseline="Corrupted (pretrained)"):
    out = {}
    for m in mitigations:
        out[m] = lift(per_case, m, baseline)
    return out

g_lifts = collect_lifts(gauss_pc, mitigations_gauss)
b_lifts = collect_lifts(bias_pc,  mitigations_bias )

all_cats = ["Filter mean_3", "Filter gaussian_1p0", "Filter N4", "Corrupted (fine-tuned)"]
fig, ax = plt.subplots(figsize=(11, 5.5))
xs = np.arange(len(all_cats))
W = 0.4
g_means = [g_lifts.get(c, (np.nan, np.nan))[0] for c in all_cats]
g_stds  = [g_lifts.get(c, (np.nan, np.nan))[1] for c in all_cats]
b_means = [b_lifts.get(c, (np.nan, np.nan))[0] for c in all_cats]
b_stds  = [b_lifts.get(c, (np.nan, np.nan))[1] for c in all_cats]
ax.bar(xs - W/2, g_means, W, yerr=g_stds, color="#d62728", alpha=0.85,
       label=f"Gaussian noise σ={sigma_chosen}", edgecolor="black", linewidth=0.5, capsize=3)
ax.bar(xs + W/2, b_means, W, yerr=b_stds, color="#9467bd", alpha=0.85,
       label=f"Bias field strength={strength_chosen}", edgecolor="black", linewidth=0.5, capsize=3)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(xs); ax.set_xticklabels([LABEL_MAP[c] for c in all_cats], fontsize=11)
ax.set_ylabel("WT-DSC lift over raw corrupted (mean ± std, paired per-patient)")
ax.set_title("How much does each mitigation actually recover?")
ax.grid(alpha=0.3, axis="y"); ax.legend(loc="upper left", fontsize=11)
for x, m in zip(xs - W/2, g_means):
    if not np.isnan(m):
        ax.text(x, m + 0.005, f"{m:+.3f}", ha="center", va="bottom", fontsize=8.5)
for x, m in zip(xs + W/2, b_means):
    if not np.isnan(m):
        ax.text(x, m + 0.005, f"{m:+.3f}", ha="center", va="bottom", fontsize=8.5)
fig.tight_layout()
fig.savefig(CFG.out_root / "figures" / "comparison_lift_per_mitigation.png", dpi=160, bbox_inches="tight")
plt.show()


## Interpretation and limits

Read actual rerun outputs rather than expecting historical results. Paired comparisons include
bootstrap CIs for mean DSC differences and Holm-adjusted Wilcoxon p-values per study.
No significant difference means insufficient evidence of a difference, not equivalence.

Each fine-tuned model is evaluated on its matching corruption and clean images. Cross-corruption
and unknown-corruption robustness are not established. Only one architecture and synthetic
artifacts are evaluated; clinical generalization and pretraining independence remain unverified.


In [ ]:
# Auto-generated summary of headline numbers from the actual run
def hd(agg_wt, cond):
    if cond not in agg_wt.index:
        return np.nan
    return float(agg_wt.loc[cond, "DSC_mean"])

bias_wt_idx  = test_agg[test_agg.region == "WT"].set_index("condition")
gauss_wt_idx = gauss_agg[gauss_agg.region == "WT"].set_index("condition")

print(f"================ HEADLINE WT-DSC SUMMARY ================")
print(f"Test patients: n = {len(test_df)}")
print()
print(f"--- Gaussian noise (σ={sigma_chosen}) ---")
print(f"  Clean (pretrained)        = {hd(gauss_wt_idx, 'Clean (pretrained)'):.3f}")
print(f"  Corrupted (pretrained)    = {hd(gauss_wt_idx, 'Corrupted (pretrained)'):.3f}  (drop)")
print(f"  Filter mean_3             = {hd(gauss_wt_idx, 'Filter mean_3'):.3f}  (example linear filter)")
print(f"  Filter N4                 = not evaluated (not evaluated in this study)")
print(f"  Fine-tuned (corrupted)    = {hd(gauss_wt_idx, 'Corrupted (fine-tuned)'):.3f}")
print(f"  Fine-tuned (clean)        = {hd(gauss_wt_idx, 'Clean (fine-tuned)'):.3f}  (forgetting check)")
print()
print(f"--- Bias field (strength={strength_chosen}) ---")
print(f"  Clean (pretrained)        = {hd(bias_wt_idx,  'Clean (pretrained)'):.3f}")
print(f"  Corrupted (pretrained)    = {hd(bias_wt_idx,  'Corrupted (pretrained)'):.3f}  (drop)")
print(f"  Filter mean_3             = {hd(bias_wt_idx,  'Filter mean_3'):.3f}  (linear filter)")
print(f"  Filter N4                 = {hd(bias_wt_idx,  'Filter N4'):.3f}  (artifact-specific preprocessing)")
print(f"  Fine-tuned (corrupted)    = {hd(bias_wt_idx,  'Corrupted (fine-tuned)'):.3f}")
print(f"  Fine-tuned (clean)        = {hd(bias_wt_idx,  'Clean (fine-tuned)'):.3f}  (forgetting check)")
print()
print(f"Wilcoxon (paired, two-sided, WT-DSC):")
for r in wcx_rows:
    print(f"  {r['comparison']:55s}  median Δ = {r['median_diff_A_minus_B']:+.4f}  raw p = {r['p_value_two_sided']:.2e}; Holm p = {r['p_value_holm']:.2e}")
